# Валидация и обучение тестовой модели

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

DATA_PATH = Path("../../data/clean/new_sales_data.csv")
RANDOM_STATE = 42


Валидация датасета ✅

In [3]:
def load_monthly_sales(csv_path: str | Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    df["sku"] = df["sku"].fillna("").astype(str).str.strip()
    df["product"] = df["product"].fillna("").astype(str).str.strip()
    df["unit"] = df["unit"].fillna("").astype(str).str.strip()
    df["month"] = pd.to_datetime(df["month"], format="%Y-%m")
    df["qty"] = pd.to_numeric(df["qty"], errors="coerce").fillna(0.0)

    # Для MVP убираем строки без SKU
    df = df.loc[df["sku"] != ""].copy()

    # Агрегируем до уровня sku-month
    monthly = (
        df.groupby(["sku", "month"], as_index=False)
        .agg(
            product=("product", "first"),
            unit=("unit", "first"),
            qty=("qty", "sum"),
        )
        .sort_values(["sku", "month"])
        .reset_index(drop=True)
    )
    return monthly


def expand_to_month_grid(monthly: pd.DataFrame) -> pd.DataFrame:
    all_months = pd.date_range(monthly["month"].min(), monthly["month"].max(), freq="MS")
    sku_meta = monthly[["sku", "product", "unit"]].drop_duplicates("sku")

    full_index = pd.MultiIndex.from_product(
        [sku_meta["sku"].tolist(), all_months],
        names=["sku", "month"],
    )

    expanded = (
        pd.DataFrame(index=full_index)
        .reset_index()
        .merge(sku_meta, on="sku", how="left")
        .merge(monthly[["sku", "month", "qty"]], on=["sku", "month"], how="left")
    )

    # Если SKU в каком-то месяце не продавался, считаем спрос = 0
    expanded["qty"] = expanded["qty"].fillna(0.0)
    expanded = expanded.sort_values(["sku", "month"]).reset_index(drop=True)
    return expanded


Добавление признаков для модели ✅

In [4]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().sort_values(["sku", "month"]).reset_index(drop=True)
    grp = df.groupby("sku", sort=False)["qty"]
    history = grp.shift(1)

    # Лаги
    for lag in [1, 2, 3, 6, 12]:
        df[f"lag_{lag}"] = grp.shift(lag)

    # Скользящие статистики только по прошлым месяцам
    df["ma3"] = history.rolling(3).mean().reset_index(level=0, drop=True)
    df["ma6"] = history.rolling(6).mean().reset_index(level=0, drop=True)
    df["rolling_std_3"] = history.rolling(3).std().reset_index(level=0, drop=True)
    df["rolling_std_6"] = history.rolling(6).std().reset_index(level=0, drop=True)

    # Разности
    df["diff_1"] = grp.diff(1).shift(1)
    df["diff_12"] = grp.diff(12).shift(1)

    # Сезонность
    df["month_num"] = df["month"].dt.month # type: ignore
    angle = 2 * np.pi * df["month_num"] / 12
    df["month_sin"] = np.sin(angle)
    df["month_cos"] = np.cos(angle)
    df["is_december"] = (df["month_num"] == 12).astype(int)

    # Простое кодирование unit
    unit_dummies = pd.get_dummies(df["unit"], prefix="unit", dtype=int)
    df = pd.concat([df, unit_dummies], axis=1)

    for col in ["unit_кг", "unit_шт"]:
        if col not in df.columns:
            df[col] = 0

    return df


Применение к данным ✅

In [5]:
monthly = load_monthly_sales(DATA_PATH)
base_df = expand_to_month_grid(monthly)
feature_df = add_features(base_df)

print("monthly shape:", monthly.shape)
print("base_df shape:", base_df.shape)
print("feature_df shape:", feature_df.shape)

feature_df


monthly shape: (16066, 5)
base_df shape: (34308, 5)
feature_df shape: (34308, 22)


,sku,month,product,unit,qty,lag_1,lag_2,lag_3,lag_6,lag_12,ma3,ma6,rolling_std_3,rolling_std_6,diff_1,diff_12,month_num,month_sin,month_cos,is_december,unit_кг,unit_шт
0,0104,2023-01-01,"УксусБассо Balsamic 0,5 л",шт,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,5.000000e-01,8.660254e-01,0,0,1
1,0104,2023-02-01,"УксусБассо Balsamic 0,5 л",шт,2.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,8.660254e-01,5.000000e-01,0,0,1
2,0104,2023-03-01,"УксусБассо Balsamic 0,5 л",шт,3.0,2.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-3.0,NaN,3,1.000000e+00,6.123234e-17,0,0,1
3,0104,2023-04-01,"УксусБассо Balsamic 0,5 л",шт,1.0,3.0,2.0,5.0,NaN,NaN,3.333333,NaN,1.527525,NaN,1.0,NaN,4,8.660254e-01,-5.000000e-01,0,0,1
4,0104,2023-05-01,"УксусБассо Balsamic 0,5 л",шт,0.0,1.0,3.0,2.0,NaN,NaN,2.000000,NaN,1.000000,NaN,-2.0,NaN,5,5.000000e-01,-8.660254e-01,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34303,слв6006,2025-08-01,Кофе сублимированный А.П. СЕЛИВАНОВ Venice Ver...,шт,2.0,5.0,5.0,1.0,0.0,0.0,3.666667,2.500000,2.309401,2.073644,0.0,5.0,8,-8.660254e-01,-5.000000e-01,0,0,1
34304,слв6006,2025-09-01,Кофе сублимированный А.П. СЕЛИВАНОВ Venice Ver...,шт,4.0,2.0,5.0,5.0,2.0,0.0,4.000000,2.833333,1.732051,1.722401,-3.0,2.0,9,-1.000000e+00,-1.836970e-16,0,0,1
34305,слв6006,2025-10-01,Кофе сублимированный А.П. СЕЛИВАНОВ Venice Ver...,шт,5.0,4.0,2.0,5.0,2.0,0.0,3.666667,3.166667,1.527525,1.722401,2.0,4.0,10,-8.660254e-01,5.000000e-01,0,0,1
34306,слв6006,2025-11-01,Кофе сублимированный А.П. СЕЛИВАНОВ Venice Ver...,шт,4.0,5.0,4.0,2.0,1.0,6.0,3.666667,3.666667,1.527525,1.751190,1.0,5.0,11,-5.000000e-01,8.660254e-01,0,0,1


In [6]:
# Cell 5
# Оставляем SKU с достаточной историей
MIN_HISTORY = 24

history_count = feature_df.groupby("sku")["qty"].transform("size")
model_df = feature_df.loc[history_count >= MIN_HISTORY].copy()

# Для lag_12 нужна история минимум 12 месяцев
model_df = model_df.dropna(subset=["lag_12"]).reset_index(drop=True)

FEATURE_COLUMNS = [
    "lag_1", "lag_2", "lag_3", "lag_6", "lag_12",
    "ma3", "ma6",
    "rolling_std_3", "rolling_std_6",
    "diff_1", "diff_12",
    "month_num", "month_sin", "month_cos", "is_december",
    "unit_кг", "unit_шт",
]

TARGET_COLUMN = "qty"

print("model_df shape:", model_df.shape)
print("sku count:", model_df["sku"].nunique())
print("months:", model_df["month"].min(), "->", model_df["month"].max())


model_df shape: (22872, 22)
sku count: 953
months: 2024-01-01 00:00:00 -> 2025-12-01 00:00:00


2023 год пропал, так как мы удалили все sku, где нет lag_12 (данных год назад)

## Старт ML
Выбираем несколько sku, которые будем прогнозировать

In [7]:
# Выбираем 5-7 SKU не совсем случайно, а с простым покрытием по объему и unit
def choose_sku_subset(df: pd.DataFrame, n_skus: int = 6, random_state: int = 42) -> list[str]:
    sku_summary = (
        df.groupby("sku", as_index=False)
        .agg(
            product=("product", "first"),
            unit=("unit", "first"),
            total_qty=("qty", "sum"),
            mean_qty=("qty", "mean"),
            non_zero_share=("qty", lambda s: (s > 0).mean()),
        )
        .sort_values("total_qty", ascending=False)
        .reset_index(drop=True)
    )

    sku_summary["volume_bucket"] = pd.qcut(
        sku_summary["total_qty"].rank(method="first"),
        q=min(3, len(sku_summary)),
        labels=["low", "mid", "high"][:min(3, len(sku_summary))],
    )

    chosen = []
    rng = np.random.default_rng(random_state)

    for _, bucket in sku_summary.groupby(["unit", "volume_bucket"], dropna=False):
        if len(chosen) >= n_skus:
            break
        take = min(2, len(bucket), n_skus - len(chosen))
        sampled = bucket.sample(n=take, random_state=int(rng.integers(0, 1_000_000)))
        chosen.extend(sampled["sku"].tolist())

    if len(chosen) < n_skus:
        extra = sku_summary.loc[~sku_summary["sku"].isin(chosen), "sku"].head(n_skus - len(chosen))
        chosen.extend(extra.tolist())

    return chosen[:n_skus]


subset_skus = choose_sku_subset(model_df, n_skus=6, random_state=RANDOM_STATE)
subset_df = model_df.loc[model_df["sku"].isin(subset_skus)].copy()

print("subset_skus:", subset_skus)
subset_df[["sku", "product", "unit"]].drop_duplicates().sort_values("sku")


subset_skus: ['РФ23890', 'РФ10619', 'НС02055', 'КО17611', 'ЙО20155', 'РФ19054']


,sku,product,unit
4080,ЙО20155,КОНФ ВЕС Птичье молоко по ГОСТ,кг
6048,КО17611,Карамель Алёнка с молочной начинкой 1 кг,кг
9264,НС02055,КОНФ ВЕС Новосибирские,кг
12168,РФ10619,КОНФ ВЕС Солнышко с семечками,кг
14256,РФ19054,Зефир СЛАДКИЕ ИСТОРИИ с черной смородиной в шо...,кг
16152,РФ23890,Карамель PUMPKINS со вкусом шоколада и тыквы,кг


Делим данные на обучающие и тестовые

In [8]:
# Временной split: последние 3 месяца в test
TEST_MONTHS = 3

cutoff_month = subset_df["month"].max() - pd.DateOffset(months=TEST_MONTHS - 1)

train_df = subset_df.loc[subset_df["month"] < cutoff_month].copy()
test_df = subset_df.loc[subset_df["month"] >= cutoff_month].copy()

print("cutoff_month:", cutoff_month)
print("train months:", train_df["month"].min(), "->", train_df["month"].max())
print("test months:", test_df["month"].min(), "->", test_df["month"].max())
print("train shape:", train_df.shape)
print("test shape:", test_df.shape)


cutoff_month: 2025-10-01 00:00:00
train months: 2024-01-01 00:00:00 -> 2025-09-01 00:00:00
test months: 2025-10-01 00:00:00 -> 2025-12-01 00:00:00
train shape: (126, 22)
test shape: (18, 22)


## RandomForestRegressor

In [9]:
model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=3,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model.fit(train_df[FEATURE_COLUMNS], train_df[TARGET_COLUMN])

test_df = test_df.copy()
test_df["prediction"] = model.predict(test_df[FEATURE_COLUMNS]).clip(min=0)

test_df[["sku", "month", "qty", "prediction"]].head(20)


,sku,month,qty,prediction
4101,ЙО20155,2025-10-01,0.000,17.317865
4102,ЙО20155,2025-11-01,12.978,21.539377
4103,ЙО20155,2025-12-01,46.867,24.810164
6069,КО17611,2025-10-01,4.739,4.810576
6070,КО17611,2025-11-01,4.741,4.345723
6071,КО17611,2025-12-01,5.712,4.558717
9285,НС02055,2025-10-01,9.548,8.528153
9286,НС02055,2025-11-01,6.623,8.594636
9287,НС02055,2025-12-01,6.247,8.720248
12189,РФ10619,2025-10-01,0.000,0.000000


In [10]:
subset_df[(subset_df["sku"] == "ЙО20155") & (subset_df["month"] == "2024-09-01")]

,sku,month,product,unit,qty,lag_1,lag_2,lag_3,lag_6,lag_12,ma3,ma6,rolling_std_3,rolling_std_6,diff_1,diff_12,month_num,month_sin,month_cos,is_december,unit_кг,unit_шт
4088,ЙО20155,2024-09-01,КОНФ ВЕС Птичье молоко по ГОСТ,кг,0.0,13.892,8.591007,17.625,14.085,0.0,13.369336,15.272001,4.539619,3.966671,5.300993,5.829,9,-1.0,-1.836970e-16,0,1,0


In [11]:
def mape_non_zero(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def wape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.abs(y_true).sum()
    if denom == 0:
        return np.nan
    return np.abs(y_true - y_pred).sum() / denom * 100


y_true = test_df["qty"]
y_pred = test_df["prediction"]

metrics = {
    "MAE": mean_absolute_error(y_true, y_pred),
    "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
    "MAPE_non_zero_%": mape_non_zero(y_true, y_pred),
    "WAPE_%": wape(y_true, y_pred),
}

metrics


{'MAE': 4.5696115681184315,
 'RMSE': np.float64(7.590605786351775),
 'MAPE_non_zero_%': np.float64(37.95741011632579),
 'WAPE_%': np.float64(56.11054445779876)}

Далее смотрим качество по каждому SKU

In [12]:
# Смотрим качество по каждому SKU
sku_metrics = (
    test_df.groupby("sku")
    .apply(
        lambda g: pd.Series(
            {
                "product": g["product"].iloc[0],
                "fact_sum": g["qty"].sum(),
                "pred_sum": g["prediction"].sum(),
                "mae": mean_absolute_error(g["qty"], g["prediction"]),
                "rmse": np.sqrt(mean_squared_error(g["qty"], g["prediction"])),
                "mape_non_zero_%": mape_non_zero(g["qty"], g["prediction"]),
            }
        ),
        include_groups=False,
    )
    .reset_index()
    .sort_values("mae", ascending=False)
)

sku_metrics


,sku,product,fact_sum,pred_sum,mae,rmse,mape_non_zero_%
0,ЙО20155,КОНФ ВЕС Птичье молоко по ГОСТ,59.845,63.667406,15.978693,16.928375,56.515501
4,РФ19054,Зефир СЛАДКИЕ ИСТОРИИ с черной смородиной в шо...,45.108,43.125974,5.890038,6.517596,44.125948
5,РФ23890,Карамель PUMPKINS со вкусом шоколада и тыквы,4.028,5.399420,3.142473,3.531317,100.000000
2,НС02055,КОНФ ВЕС Новосибирские,22.418,25.843038,1.821577,1.918714,26.680591
1,КО17611,Карамель Алёнка с молочной начинкой 1 кг,15.192,13.715017,0.540045,0.705083,10.012767
3,РФ10619,КОНФ ВЕС Солнышко с семечками,0.000,0.134529,0.044843,0.077670,NaN


In [13]:
# Важность признаков
feature_importance = (
    pd.DataFrame(
        {
            "feature": FEATURE_COLUMNS,
            "importance": model.feature_importances_,
        }
    )
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

feature_importance


,feature,importance
0,lag_12,0.351181
1,ma3,0.146305
2,ma6,0.134409
3,lag_1,0.084053
4,rolling_std_6,0.068168
5,month_num,0.055269
6,diff_12,0.036384
7,lag_6,0.030270
8,lag_2,0.025474
9,lag_3,0.020251


In [14]:
# Финальная таблица прогноза для проверки
result = test_df[
    ["sku", "product", "unit", "month", "qty", "prediction"]
].sort_values(["sku", "month"]).reset_index(drop=True)

result.head(30)


,sku,product,unit,month,qty,prediction
0,ЙО20155,КОНФ ВЕС Птичье молоко по ГОСТ,кг,2025-10-01,0.000,17.317865
1,ЙО20155,КОНФ ВЕС Птичье молоко по ГОСТ,кг,2025-11-01,12.978,21.539377
2,ЙО20155,КОНФ ВЕС Птичье молоко по ГОСТ,кг,2025-12-01,46.867,24.810164
3,КО17611,Карамель Алёнка с молочной начинкой 1 кг,кг,2025-10-01,4.739,4.810576
4,КО17611,Карамель Алёнка с молочной начинкой 1 кг,кг,2025-11-01,4.741,4.345723
5,КО17611,Карамель Алёнка с молочной начинкой 1 кг,кг,2025-12-01,5.712,4.558717
6,НС02055,КОНФ ВЕС Новосибирские,кг,2025-10-01,9.548,8.528153
7,НС02055,КОНФ ВЕС Новосибирские,кг,2025-11-01,6.623,8.594636
8,НС02055,КОНФ ВЕС Новосибирские,кг,2025-12-01,6.247,8.720248
9,РФ10619,КОНФ ВЕС Солнышко с семечками,кг,2025-10-01,0.000,0.000000


## Усиление качества без смены модели

Ниже добавлен блок, который не меняет базовую модель `RandomForestRegressor`, но делает проверку качества репрезентативной:
- rolling time-series validation по всем SKU,
- минимальный feature engineering для шумных нулей,
- Optuna tuning на тех же признаках,
- абляция `lag_12`,
- анализ ошибок по SKU и SHAP.


In [ ]:
import optuna
import shap


def add_group_rolling(df: pd.DataFrame, base_col: str, new_col: str, window: int, func: str = "mean", min_periods: int | None = None) -> None:
    min_periods = window if min_periods is None else min_periods
    df[new_col] = (
        df.groupby("sku", sort=False)[base_col]
        .transform(lambda s: s.rolling(window=window, min_periods=min_periods).agg(func))
    )


def add_features_v2(df: pd.DataFrame, include_smooth: bool = True, include_lag12: bool = True) -> tuple[pd.DataFrame, list[str]]:
    df = df.copy().sort_values(["sku", "month"]).reset_index(drop=True)
    grp = df.groupby("sku", sort=False)["qty"]
    history = grp.shift(1)

    df["history"] = history
    df["history_non_zero"] = history.where(history > 0)
    df["history_is_non_zero"] = history.gt(0).astype(float)

    lag_list = [1, 2, 3, 6] + ([12] if include_lag12 else [])
    for lag in lag_list:
        df[f"lag_{lag}"] = grp.shift(lag)

    add_group_rolling(df, "history", "ma3", 3, "mean")
    add_group_rolling(df, "history", "ma6", 6, "mean")
    add_group_rolling(df, "history", "rolling_std_3", 3, "std")
    add_group_rolling(df, "history", "rolling_std_6", 6, "std")

    df["diff_1"] = grp.diff(1).shift(1)
    if include_lag12:
        df["diff_12"] = grp.diff(12).shift(1)

    df["month_num"] = df["month"].dt.month
    angle = 2 * np.pi * df["month_num"] / 12
    df["month_sin"] = np.sin(angle)
    df["month_cos"] = np.cos(angle)

    feature_cols = [
        "lag_1", "lag_2", "lag_3", "lag_6",
        "ma3", "ma6",
        "rolling_std_3", "rolling_std_6",
        "diff_1",
        "month_num", "month_sin", "month_cos",
    ]

    if include_lag12:
        feature_cols = ["lag_12", *feature_cols]
        feature_cols.insert(feature_cols.index("month_num"), "diff_12")

    if include_smooth:
        add_group_rolling(df, "history", "median3", 3, "median")
        add_group_rolling(df, "history", "median6", 6, "median")
        add_group_rolling(df, "history_non_zero", "ma3_non_zero", 3, "mean", min_periods=1)
        add_group_rolling(df, "history_is_non_zero", "non_zero_share_6", 6, "mean", min_periods=1)

        base = df["ma3_non_zero"].fillna(df["ma3"]).fillna(0)
        df["lag_1_to_ma3"] = (df["lag_1"] / (base + 1.0)).clip(-5, 5)

        feature_cols = [
            *(["lag_12"] if include_lag12 else []),
            "lag_1", "lag_2", "lag_3", "lag_6",
            "ma3", "ma6", "median3", "median6", "ma3_non_zero",
            "rolling_std_3", "rolling_std_6", "non_zero_share_6",
            "diff_1",
            *(["diff_12"] if include_lag12 else []),
            "lag_1_to_ma3",
            "month_num", "month_sin", "month_cos",
        ]

    return df, feature_cols


def build_model_df_v2(include_smooth: bool = True, include_lag12: bool = True, min_history: int = 24) -> tuple[pd.DataFrame, list[str]]:
    monthly_local = load_monthly_sales(DATA_PATH)
    base_df_local = expand_to_month_grid(monthly_local)
    feature_df_local, feature_cols_local = add_features_v2(base_df_local, include_smooth=include_smooth, include_lag12=include_lag12)

    history_count_local = feature_df_local.groupby("sku")["qty"].transform("size")
    model_df_local = feature_df_local.loc[history_count_local >= min_history].copy()

    na_cols = ["lag_6"]
    if include_lag12:
        na_cols.append("lag_12")
    model_df_local = model_df_local.dropna(subset=na_cols).reset_index(drop=True)
    return model_df_local, feature_cols_local


def get_time_series_splits(df: pd.DataFrame, horizon: int = 3, min_train_months: int = 12, n_recent_folds: int = 4) -> list[tuple[pd.Timestamp, np.ndarray]]:
    months_local = np.array(sorted(df["month"].unique()))
    splits_local: list[tuple[pd.Timestamp, np.ndarray]] = []
    for train_end_idx in range(min_train_months - 1, len(months_local) - horizon):
        train_end_month = months_local[train_end_idx]
        valid_months = months_local[train_end_idx + 1 : train_end_idx + 1 + horizon]
        splits_local.append((train_end_month, valid_months))
    return splits_local[-n_recent_folds:]


def apply_sparse_rule(df: pd.DataFrame, pred: np.ndarray) -> np.ndarray:
    required_cols = {"non_zero_share_6", "lag_1", "lag_2", "ma3_non_zero"}
    if not required_cols.issubset(df.columns):
        return pred

    sparse_mask = (
        (df["non_zero_share_6"].fillna(0) <= 0.2)
        & (df["lag_1"].fillna(0) == 0)
        & (df["lag_2"].fillna(0) == 0)
        & (df["ma3_non_zero"].fillna(0) < 1.0)
    )
    return np.where(sparse_mask, 0.0, pred)


def evaluate_model_cv(df: pd.DataFrame, feature_cols: list[str], params: dict, use_sparse_rule: bool = False) -> tuple[pd.DataFrame, pd.DataFrame]:
    fold_rows = []
    pred_parts = []

    for fold_id, (train_end, valid_months) in enumerate(get_time_series_splits(df), start=1):
        train_part = df.loc[df["month"] <= train_end].copy()
        valid_part = df.loc[df["month"].isin(valid_months)].copy()

        model_local = RandomForestRegressor(**params)
        model_local.fit(train_part[feature_cols], train_part["qty"])

        train_pred = model_local.predict(train_part[feature_cols]).clip(min=0)
        valid_pred = model_local.predict(valid_part[feature_cols]).clip(min=0)
        if use_sparse_rule:
            valid_pred = apply_sparse_rule(valid_part, valid_pred)

        fold_rows.append(
            {
                "fold": fold_id,
                "train_wape": wape(train_part["qty"], train_pred),
                "valid_wape": wape(valid_part["qty"], valid_pred),
                "train_mape_non_zero": mape_non_zero(train_part["qty"], train_pred),
                "valid_mape_non_zero": mape_non_zero(valid_part["qty"], valid_pred),
                "valid_mae": mean_absolute_error(valid_part["qty"], valid_pred),
                "valid_rmse": float(np.sqrt(mean_squared_error(valid_part["qty"], valid_pred))),
            }
        )

        pred_part = valid_part[["sku", "product", "unit", "month", "qty"]].copy()
        pred_part["prediction"] = valid_pred
        pred_part["fold"] = fold_id
        pred_parts.append(pred_part)

    return pd.DataFrame(fold_rows), pd.concat(pred_parts, ignore_index=True)


In [ ]:
legacy_model_df, legacy_feature_cols = build_model_df_v2(include_smooth=False, include_lag12=True)
optimized_model_df, optimized_feature_cols = build_model_df_v2(include_smooth=True, include_lag12=True)

baseline_params_v2 = {
    "n_estimators": 300,
    "max_depth": 12,
    "min_samples_leaf": 3,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

legacy_cv, _ = evaluate_model_cv(legacy_model_df, legacy_feature_cols, baseline_params_v2)
optimized_default_cv, _ = evaluate_model_cv(optimized_model_df, optimized_feature_cols, baseline_params_v2)

feature_comparison = pd.DataFrame(
    [
        {
            "model": "legacy_default",
            "features": len(legacy_feature_cols),
            "cv_wape_mean": legacy_cv["valid_wape"].mean(),
            "cv_accuracy_wape": 100 - legacy_cv["valid_wape"].mean(),
            "cv_mape_non_zero_mean": legacy_cv["valid_mape_non_zero"].mean(),
        },
        {
            "model": "optimized_features_default",
            "features": len(optimized_feature_cols),
            "cv_wape_mean": optimized_default_cv["valid_wape"].mean(),
            "cv_accuracy_wape": 100 - optimized_default_cv["valid_wape"].mean(),
            "cv_mape_non_zero_mean": optimized_default_cv["valid_mape_non_zero"].mean(),
        },
    ]
)

feature_comparison.sort_values("cv_wape_mean")


### Aggressive Optuna Tuning

Тюнинг идёт на той же модели `RandomForestRegressor` и на rolling time-series split. Цель оптимизации — снизить `WAPE`, а не подогнать holdout из 6 SKU.


In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective_rf(trial: optuna.trial.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "max_depth": trial.suggest_int("max_depth", 5, 18),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 12),
        "max_features": trial.suggest_float("max_features", 0.35, 1.0),
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    }

    fold_scores = []
    for train_end, valid_months in get_time_series_splits(optimized_model_df):
        train_part = optimized_model_df.loc[optimized_model_df["month"] <= train_end].copy()
        valid_part = optimized_model_df.loc[optimized_model_df["month"].isin(valid_months)].copy()

        model_local = RandomForestRegressor(**params)
        model_local.fit(train_part[optimized_feature_cols], train_part["qty"])
        valid_pred = model_local.predict(valid_part[optimized_feature_cols]).clip(min=0)
        fold_scores.append(wape(valid_part["qty"], valid_pred))

    return float(np.mean(fold_scores))


study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective_rf, n_trials=100, show_progress_bar=False)

study.best_params


In [ ]:
best_params = {
    **study.best_params,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

optimized_tuned_cv, optimized_tuned_pred = evaluate_model_cv(
    optimized_model_df,
    optimized_feature_cols,
    best_params,
)

optimized_tuned_rule_cv, optimized_tuned_rule_pred = evaluate_model_cv(
    optimized_model_df,
    optimized_feature_cols,
    best_params,
    use_sparse_rule=True,
)

optimized_no_lag12_df, optimized_no_lag12_cols = build_model_df_v2(include_smooth=True, include_lag12=False)
optimized_no_lag12_cv, _ = evaluate_model_cv(
    optimized_no_lag12_df,
    optimized_no_lag12_cols,
    best_params,
)

cv_summary = pd.DataFrame(
    [
        {
            "model": "legacy_default",
            "cv_wape_mean": legacy_cv["valid_wape"].mean(),
            "cv_accuracy_wape": 100 - legacy_cv["valid_wape"].mean(),
            "cv_mape_non_zero_mean": legacy_cv["valid_mape_non_zero"].mean(),
            "train_wape_mean": legacy_cv["train_wape"].mean(),
        },
        {
            "model": "optimized_default",
            "cv_wape_mean": optimized_default_cv["valid_wape"].mean(),
            "cv_accuracy_wape": 100 - optimized_default_cv["valid_wape"].mean(),
            "cv_mape_non_zero_mean": optimized_default_cv["valid_mape_non_zero"].mean(),
            "train_wape_mean": optimized_default_cv["train_wape"].mean(),
        },
        {
            "model": "optimized_tuned_lag12",
            "cv_wape_mean": optimized_tuned_cv["valid_wape"].mean(),
            "cv_accuracy_wape": 100 - optimized_tuned_cv["valid_wape"].mean(),
            "cv_mape_non_zero_mean": optimized_tuned_cv["valid_mape_non_zero"].mean(),
            "train_wape_mean": optimized_tuned_cv["train_wape"].mean(),
        },
        {
            "model": "optimized_tuned_no_lag12",
            "cv_wape_mean": optimized_no_lag12_cv["valid_wape"].mean(),
            "cv_accuracy_wape": 100 - optimized_no_lag12_cv["valid_wape"].mean(),
            "cv_mape_non_zero_mean": optimized_no_lag12_cv["valid_mape_non_zero"].mean(),
            "train_wape_mean": optimized_no_lag12_cv["train_wape"].mean(),
        },
        {
            "model": "optimized_tuned_lag12_sparse_rule",
            "cv_wape_mean": optimized_tuned_rule_cv["valid_wape"].mean(),
            "cv_accuracy_wape": 100 - optimized_tuned_rule_cv["valid_wape"].mean(),
            "cv_mape_non_zero_mean": optimized_tuned_rule_cv["valid_mape_non_zero"].mean(),
            "train_wape_mean": optimized_tuned_rule_cv["train_wape"].mean(),
        },
    ]
).sort_values("cv_wape_mean")

cv_summary


In [ ]:
lag12_ablation = pd.DataFrame(
    [
        {
            "variant": "with_lag12",
            "cv_wape_mean": optimized_tuned_cv["valid_wape"].mean(),
            "cv_accuracy_wape": 100 - optimized_tuned_cv["valid_wape"].mean(),
            "cv_mape_non_zero_mean": optimized_tuned_cv["valid_mape_non_zero"].mean(),
        },
        {
            "variant": "without_lag12",
            "cv_wape_mean": optimized_no_lag12_cv["valid_wape"].mean(),
            "cv_accuracy_wape": 100 - optimized_no_lag12_cv["valid_wape"].mean(),
            "cv_mape_non_zero_mean": optimized_no_lag12_cv["valid_mape_non_zero"].mean(),
        },
    ]
)

lag12_ablation


In [ ]:
subset_skus_v2 = choose_sku_subset(optimized_model_df, n_skus=6, random_state=RANDOM_STATE)
subset_df_v2 = optimized_model_df.loc[optimized_model_df["sku"].isin(subset_skus_v2)].copy()
cutoff_month_v2 = subset_df_v2["month"].max() - pd.DateOffset(months=2)
train_subset_v2 = subset_df_v2.loc[subset_df_v2["month"] < cutoff_month_v2].copy()
test_subset_v2 = subset_df_v2.loc[subset_df_v2["month"] >= cutoff_month_v2].copy()

subset_results = []
for model_name, params_local, feature_cols_local, use_sparse_rule_local in [
    ("legacy_default", baseline_params_v2, legacy_feature_cols, False),
    ("optimized_default", baseline_params_v2, optimized_feature_cols, False),
    ("optimized_tuned_lag12", best_params, optimized_feature_cols, False),
    ("optimized_tuned_lag12_sparse_rule", best_params, optimized_feature_cols, True),
]:
    model_local = RandomForestRegressor(**params_local)
    model_local.fit(train_subset_v2[feature_cols_local], train_subset_v2["qty"])
    pred_local = model_local.predict(test_subset_v2[feature_cols_local]).clip(min=0)
    if use_sparse_rule_local:
        pred_local = apply_sparse_rule(test_subset_v2, pred_local)

    subset_results.append(
        {
            "model": model_name,
            "MAE": mean_absolute_error(test_subset_v2["qty"], pred_local),
            "RMSE": float(np.sqrt(mean_squared_error(test_subset_v2["qty"], pred_local))),
            "MAPE_non_zero_%": mape_non_zero(test_subset_v2["qty"], pred_local),
            "WAPE_%": wape(test_subset_v2["qty"], pred_local),
        }
    )

subset_summary = pd.DataFrame(subset_results)
subset_summary


In [ ]:
sku_metrics_tuned = (
    optimized_tuned_pred.groupby("sku")
    .apply(
        lambda g: pd.Series(
            {
                "product": g["product"].iloc[0],
                "fact_sum": g["qty"].sum(),
                "pred_sum": g["prediction"].sum(),
                "wape": wape(g["qty"], g["prediction"]),
                "mape_non_zero": mape_non_zero(g["qty"], g["prediction"]),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)

sku_level = (
    optimized_model_df.groupby("sku")
    .agg(
        avg_qty=("qty", "mean"),
        zero_share_all=("qty", lambda s: (s == 0).mean()),
        std_qty=("qty", "std"),
    )
    .fillna({"std_qty": 0})
    .reset_index()
)

sku_metrics_tuned = sku_metrics_tuned.merge(sku_level, on="sku", how="left")
sku_metrics_tuned["volume_bucket"] = pd.qcut(
    sku_metrics_tuned["avg_qty"].rank(method="first"),
    q=3,
    labels=["low", "mid", "high"],
)
sku_metrics_tuned["zero_bucket"] = pd.cut(
    sku_metrics_tuned["zero_share_all"],
    bins=[-0.01, 0.05, 0.3, 1.0],
    labels=["dense", "mixed", "sparse"],
)

segment_summary = (
    sku_metrics_tuned.groupby(["volume_bucket", "zero_bucket"], observed=False)
    .agg(
        sku_count=("sku", "count"),
        mean_wape=("wape", "mean"),
        median_wape=("wape", "median"),
    )
    .reset_index()
)

worst_sku = sku_metrics_tuned.sort_values("wape", ascending=False).head(15)
best_sku = sku_metrics_tuned.sort_values("wape", ascending=True).head(15)

segment_summary


In [ ]:
latest_train_end, latest_valid_months = get_time_series_splits(optimized_model_df)[-1]
train_latest = optimized_model_df.loc[optimized_model_df["month"] <= latest_train_end].copy()
valid_latest = optimized_model_df.loc[optimized_model_df["month"].isin(latest_valid_months)].copy()

final_model = RandomForestRegressor(**best_params)
final_model.fit(train_latest[optimized_feature_cols], train_latest["qty"])

valid_sample = valid_latest[optimized_feature_cols].sample(
    n=min(800, len(valid_latest)),
    random_state=RANDOM_STATE,
)

explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(valid_sample)

shap_summary = (
    pd.DataFrame(
        {
            "feature": optimized_feature_cols,
            "mean_abs_shap": np.abs(shap_values).mean(axis=0),
            "feature_importance": final_model.feature_importances_,
        }
    )
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

weak_features = shap_summary.loc[
    (shap_summary["mean_abs_shap"] < shap_summary["mean_abs_shap"].median() * 0.25)
    & (shap_summary["feature_importance"] < 0.02),
    "feature",
].tolist()

shap_summary


### Краткие выводы по улучшениям

- Если смотреть на rolling CV по всему каталогу, лучшая версия — сглаженные признаки + Optuna + простой sparse-rule для товаров с длинной серией нулей.
- `lag_12` полезен, но эффект маленький: он даёт небольшой плюс, а не основной прирост.
- Главный буст пришёл не от новых лагов, а от более устойчивых признаков истории (`median`, `mean без нулей`, `lag / ma3`) и от сильной регуляризации леса.
- Если смотреть только на holdout из 6 SKU, картина может быть другой: маленькая выборка переоценивает более сложные деревья. Поэтому ориентируемся на rolling CV, а subset оставляем как ручную sanity-check проверку.
